# Cleaning and Transformation of merged Dataset

This Jupyter Notebook builds up on the result of *explore_indicator_data.ipynb* where a first data cleaning was done. The 4 original datasets were merged on the smallest (bulk density dataset) to a dataset with all columns of interest combined, as they are needed to prepare SHI calculation.

**! ! ! Run *explore_indicator_data.ipynb* first to receive the result *df_all_coi_merged.csv* in the output folder. Otherwise this Notebook won't work ! ! !**

### CONTENT
1) Data Cleaning
    + reduce to neccessary columns
    + turn empty cells into NaN

2) Data Transformation
    + rename and refill Depth & EC columns
    + check datatypes
    + rename all columns to have units
    + change '< LOD' values according to agreement (random numbers between 0 and min)
    + add, calculate and transform all columns
    + calculate quantiles for OC and Clay:SOC ratio
    + use quantiles and predefined bins to determine a score for each indicator in each sample


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Data Cleaning
+ reduce to neccessary columns
+ turn empty cells into NaN


In [ ]:
# load the result result of *explore_indicator_data.ipynb*
df_all_coi_merged = pd.read_csv("../output/df_all_coi_merged.csv")

# drop unused columns for better overview, not necessary
df_all_coi_merged.drop(columns=['LC', 'LU', 'LC0_Desc', 'LC1_Desc', 'LU1_Desc', 'TH_LAT', 'TH_LONG'], inplace=True)

# replace all missing values with NaN
df_all_coi_merged.replace('', np.nan, inplace=True)

# delete all rows that have NaNs and copy all kept to a new df
df_without_nan = df_all_coi_merged.dropna().copy()

# print to check
print("Dataset Size after deleting NaN:", len(df_without_nan))
df_without_nan.head()

# save the df
df_without_nan.to_csv("../output/df_without_nan.csv", index=False)

# check
df_without_nan.head()


## 2. Data Transformation

+ rename and refill Depth & EC columns
+ delete rows with missing or wrong values
+ rename all columns to have units
+ change '< LOD' values according to agreement (random number)
+ add and calculate and transform columns
+  update datatypes
+ calculate quantils (...)

### Transform Depth

In [ ]:
# for cell independence load df_without_nan from csv
df_without_nan = pd.read_csv("../output/df_without_nan.csv")

# keep only data samples in the dataframe where Depth contains '0-20 cm'
df_without_nan = df_without_nan[df_without_nan['Depth'] == '0-20 cm']

# make a new depth column and fill it with integer 20, as Depth: '0-20 cm' is a string
df_without_nan["depth_cm"] = 20

# drop the now useless Depth column
df_without_nan.drop(columns=['Depth'], inplace=True)

print("Dataset Size after deleting wrong Depths:", len(df_without_nan))
df_without_nan.head()

### Handle '< LOD' values

#### Result of exploration:
- OC LOD = 2.0, 0 samples below, '< LOD' string in 5 samples
- P LOD = 10.0, 32 samples below, '< LOD' string in 1711 samples
- N LOD = 0.2, 0 samples below, '< LOD' string in 2 samples
- K LOD = 10.0, 2 samples below, '< LOD' string in 24 samples

+ '< LOD' occur in 4 columns (OC, N, P, K)
+ for following calculations these values need to be numeric
+ approach:
    + export the current dataframe *df_without_nan* as *df_without_nan_lod_orig.csv*, make a working copy of the dataframe *df_without_nan* called *df_transform*
    + assign to each '< LOD' occurence a random number between 0 and the minimum of that indicator
    + keep in mind: in some cases the minimum itself is lower than LOD which is a contradiction, keep the contradicting samples

In [ ]:
df_without_nan.to_csv("../output/df_without_nan_lod_orig.csv", index=False)
df_transform_0 = df_without_nan.copy()

# create a dictionary for the columns with '< LOD' and their specific min value (see explore_indicator_data.ipynb)
# different approaches are possible: mean of median of all values below official LOD (...)
indicator_min_dict = {
    'OC': 2.2,
    'N': 0.2,
    'P': 0.3,
    'K': 6.2
}

# Set a seed for reproducibility
np.random.seed(28)

# Loop through each column
for indicator, min in indicator_min_dict.items():
    df_transform_0[indicator] = df_transform_0[indicator].apply(
        lambda x: round(np.random.uniform(0, min), 1) if x == '< LOD' else float(x)
    )

# SANITY CHECK - DF SHOULD BE CLEAN NOW
# Check for NaN values in the entire DataFrame
nan_counts = df_transform_0.isna().sum().sum()
print("Number of NaNs in the DataFrame: ", nan_counts)
# Check for empty strings in the entire DataFrame
empty_counts = (df_transform_0 == '').sum().sum()
print("Number of empty strings in the DataFrame: ", empty_counts)
# Count all cells that contain '< LOD' in the entire DataFrame
total_lod = (df_transform_0.astype(str).apply(lambda col: col.str.contains('< LOD'))).sum().sum()
print("Total number of '< LOD' cells in the DataFrame:", total_lod)
# check negative values
neg_values_present = (df_transform_0.select_dtypes(include='number') < 0).any().any()
print("Negative values in DataFrame? ", neg_values_present)

print("Dataset Size:", len(df_transform_0))

# save
df_transform_0.to_csv("../output/df_transform_0.csv", index=False)

# Check the result
df_transform_0.head()

### Check datatypes
As the dataframe has by now only numeric values, Pandas automatically applied the correct datatypes. We do not need to transform explicitly.
### Transform EC and Erosion, rename columns (assign units)

In [ ]:
# for cell independence load df from csv
df_transform_1 = pd.read_csv("../output/df_transform_0.csv")

# check dtypes, should be int or float
# print(df_transform_1.dtypes)

# we need EC in deciSiemens per meter, it comes in milliSiemens per meter, so /100 or * 0.01
df_transform_1["EC"] = (df_transform_1["EC"]/100).round(4)

# round erosion_by_water to only 2 decimal places
df_transform_1["erosion_by_water"] = (df_transform_1["erosion_by_water"]).round(2)


# Rename columns, integrate units for clarity
df_transform_1.rename(columns={"BD 0-20": "bulk_density_g/cm3"}, inplace=True)
df_transform_1.rename(columns={"EC": "EC_dS/m"}, inplace=True)
df_transform_1.rename(columns={"OC": "OC_g/kg"}, inplace=True)
df_transform_1.rename(columns={"P": "P_mg/kg"}, inplace=True)
df_transform_1.rename(columns={"N": "N_g/kg"}, inplace=True)
df_transform_1.rename(columns={"K": "K_mg/kg"}, inplace=True)
df_transform_1.rename(columns={"Clay": "Clay_pct"}, inplace=True)
df_transform_1.rename(columns={"erosion_by_water": "erosion_by_water_t/ha/yr"}, inplace=True)

# save
df_transform_1.to_csv("../output/df_transform_1.csv", index=False)

# Check the result
df_transform_1.head()

### Calculation of N, P, K stocks in kg/ha and the Clay:SOC ratio
- defining functions
- calculation of the result columns
- rounding of the results

In [ ]:
# for cell independence load df from csv
df_transform_2 = pd.read_csv("../output/df_transform_1.csv")

# defining a function to calculate the stock values in kg/ha for N, P and K
# it takes the value in the column (N, P or K) as concentration, bulk_density and depth as itself and unit has to be entered!
# for N it is g/kg and for P and K it is mg/kg
# the returned value is always in kg/ha (N, P or K)

def calc_stock(concentration, bulk_density, depth, unit):
    factors = {
        "mg/kg": 0.1,
        "g/kg": 100
    }
    
    if unit not in factors:
        raise ValueError(f"Unit '{unit}' not recognized. Use 'mg/kg' or 'g/kg'.")

    stock = (concentration * bulk_density * depth * factors[unit]).round(4)
    return stock

# function returns a number that represents the ratio of clay to SOC
# we multiply by 10 as OC comes in g/kg which is 0,1% but we want 1% as Clay also comes in %, it is the same as doing clay / (soc : 10)
# in case soc is 0, we cannot divide by 0, NaN would be the result
def calc_clay_SOC_ratio (clay, soc):
    ratio = np.where(soc == 0, np.nan, (clay / soc * 10).round(4))
    return ratio

# calculate the stocks for each element using the function calc_stock
df_transform_2["P_kg/ha"] = calc_stock(
    df_transform_2["P_mg/kg"],
    df_transform_2["bulk_density_g/cm3"],
    df_transform_2["depth_cm"],
    unit="mg/kg"
)

df_transform_2["N_kg/ha"] = calc_stock(
    df_transform_2["N_g/kg"],
    df_transform_2["bulk_density_g/cm3"],
    df_transform_2["depth_cm"],
    unit="g/kg"
)

df_transform_2["K_kg/ha"] = calc_stock(
    df_transform_2["K_mg/kg"],
    df_transform_2["bulk_density_g/cm3"],
    df_transform_2["depth_cm"],
    unit="mg/kg"
)

df_transform_2["Clay_SOC_ratio"] = calc_clay_SOC_ratio(
    df_transform_2["Clay_pct"],
    df_transform_2["OC_g/kg"]
)

# rounding the new columns to 2 digits
cols_to_round = ["N_kg/ha", "P_kg/ha", "K_kg/ha", "Clay_SOC_ratio"]
df_transform_2[cols_to_round] = df_transform_2[cols_to_round].round(2)

# save
df_transform_2.to_csv("../output/df_transform_2.csv", index=False)

# check
df_transform_2.head()

### Quantiles & Bins
In order to assign scores to each samples' indicators, a scale or bins are needed.
- Calculation of quantiles for 'Clay_SOC_ratio' and 'OC_g/kg' to use as bins
- Introduction of predefined bins for ('bulk_density_g/cm3', 'pH_CaCl2', 'EC_dS/m', 'OC_g/kg', 'P_kg/ha', 'N_kg/ha', 'K_kg/ha', 'erosion_by_water_t/ha/yr')
- the predefined bins are taken from the table of GSHI_Luis_Lado.pdf page 4(SHI Candidates)

In [ ]:
# for cell independence load df from csv
df_transform_3 = pd.read_csv("../output/df_transform_2.csv")

# drop unused columns for better overview
df_transform_3.drop(columns=["P_mg/kg", "N_g/kg", "K_mg/kg", "Clay_pct", "depth_cm"], inplace=True)


#### Quantiles and Scores for 'Clay_SOC_ratio' and 'OC_g/kg'

In [ ]:
# check
df_transform_3.head()

In [ ]:
# defining the quantiles
quantiles = [0.25, 0.50, 0.75]
quantile_clay_soc = df_transform_3["Clay_SOC_ratio"].quantile(quantiles)
quantile_oc = df_transform_3["OC_g/kg"].quantile(quantiles)

# Checking the quantiles for 'Clay_SOC_ratio' and 'OC_g/kg
# print(f"Quantiles for Clay_SOC_ratio:\n{quantile_clay_soc} \n")
# print(f"Quantiles for OC_g/kg:\n{quantile_oc} \n", )


# calculate the quantile scores clay_soc_ratio
# clay_soc labels are reversed: more soc = good = lower clay_soc_ratio = excellent (label 4)
quantile_labels_clay_soc = [4, 3, 2, 1]
df_transform_3["Clay_SOC_score"] = pd.qcut(df_transform_3["Clay_SOC_ratio"], q=4, labels=quantile_labels_clay_soc)

# calculate quantile scores for oc
# labels ordered: more oc = good = higher value = excellent (label 4)
quantile_labels_oc = [1, 2, 3, 4]
df_transform_3["OC_score"] = pd.qcut(df_transform_3["OC_g/kg"], q=4, labels=quantile_labels_oc)

# change dtype to int (it is category otherwise)
df_transform_3["Clay_SOC_score"] = df_transform_3["Clay_SOC_score"].astype(int)
df_transform_3["OC_score"] = df_transform_3["OC_score"].astype(int)

# check
df_transform_3.head()

#### Predefined bins for ('bulk_density_g/cm3', 'pH_CaCl2', 'EC_dS/m', 'P_kg/ha', 'N_kg/ha', 'K_kg/ha', 'erosion_by_water_t/ha/yr')

In [ ]:
# Labels ascending: P, N, K: low = 1 (poor), high = 4 (excellent)
labels_asc = [1, 2, 3, 4]
bins_P = [0, 5, 10, 20, float('inf')]  # inf means "above 20"
bins_N = [0, 50, 100, 200, float('inf')]  # inf means "above 200"
bins_K = [0, 50, 100, 200, float('inf')]  # inf means "above 200"

# Labels descending: bulk density, EC, erosion: high = 1 (poor), low = 4 (excellent)
labels_desc = [4, 3, 2, 1]
bins_bulk_density = [0, 1.2, 1.4, 1.6, float('inf')] # inf means "above 1.6"
bins_EC = [0, 1.0, 2.0, 4.0, float('inf')] # inf means "above 4.0"
bins_erosion = [0, 1.0, 2.0, 5.0, float('inf')]


# calculating the scores_labels (scores) according to bins
df_transform_3["P_score"] = pd.cut(
    df_transform_3["P_kg/ha"], 
    bins=bins_P, 
    labels=labels_asc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["N_score"] = pd.cut(
    df_transform_3["N_kg/ha"], 
    bins=bins_N, 
    labels=labels_asc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["K_score"] = pd.cut(
    df_transform_3["K_kg/ha"], 
    bins=bins_K, 
    labels=labels_asc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["bulk_density_score"] = pd.cut(
    df_transform_3["bulk_density_g/cm3"], 
    bins=bins_bulk_density, 
    labels=labels_desc, 
    right=True,    # right=True means the right edge is INCLUDED ', '
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["EC_score"] = pd.cut(
    df_transform_3["EC_dS/m"], 
    bins=bins_EC, 
    labels=labels_desc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["erosion_score"] = pd.cut(
    df_transform_3["erosion_by_water_t/ha/yr"], 
    bins=bins_erosion, 
    labels=labels_desc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

# change dtype to int (it is category otherwise)
df_transform_3["P_score"] = df_transform_3["P_score"].astype(int)
df_transform_3["N_score"] = df_transform_3["N_score"].astype(int)
df_transform_3["K_score"] = df_transform_3["K_score"].astype(int)
df_transform_3["bulk_density_score"] = df_transform_3["bulk_density_score"].astype(int)
df_transform_3["EC_score"] = df_transform_3["EC_score"].astype(int)
df_transform_3["erosion_score"] = df_transform_3["erosion_score"].astype(int)

# save
df_transform_3.to_csv("../output/df_transform_3.csv", index=False)

# check
df_transform_3.head()

#### ph-CaCl Score calculation - special bins
The nature of ph-values is that the middle values (around 7 neutral ph) are rather good, edge values (like 5 or 10) are rather poor for soil health.

In [ ]:
# for cell independence load df from csv
df_transform_4 = pd.read_csv("../output/df_transform_3.csv")

conditions = [
    (df_transform_4["pH_CaCl2"] < 5.0) | (df_transform_4["pH_CaCl2"] > 8.5),        # label 1: ph < 5.0, > 8.5
    ((df_transform_4["pH_CaCl2"] >= 5.0) & (df_transform_4["pH_CaCl2"] <= 6.0)) |   # label 2: ph 5.0 - 6.0 (includes 5.5-6.0), 7.5-8.5
    ((df_transform_4["pH_CaCl2"] >= 7.5) & (df_transform_4["pH_CaCl2"] <= 8.5)),
    ((df_transform_4["pH_CaCl2"] >= 6.0) & (df_transform_4["pH_CaCl2"] <= 6.5)) |   # label 3: ph 6.0 - 6.5, 7.0 - 7.5
    ((df_transform_4["pH_CaCl2"] >= 7.0) & (df_transform_4["pH_CaCl2"] <= 7.5)),
    ((df_transform_4["pH_CaCl2"] > 6.5) & (df_transform_4["pH_CaCl2"] < 7.0))       # label 4: ph 6.5 - 7.0
]

ph_labels = [1, 2, 3, 4]

df_transform_4["pH_score"] = np.select(conditions, ph_labels, default=np.nan)
df_transform_4["pH_score"] = df_transform_4["pH_score"].astype(int)

# save
df_transform_4.to_csv("../output/df_transform_4.csv", index=False)

# check
df_transform_4.head()